# Working With Date and time

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
date = pd.read_csv('E:\\WORK\\AI-ML\\data\\orders.csv')
time = pd.read_csv('E:\\WORK\\AI-ML\\data\\orders.csv')

In [ ]:
date.head()

In [ ]:
time.head()

In [ ]:
date.info()

In [ ]:
time.info()

# Working with Dates

In [ ]:
# Converting to datetime datatypes
date['date'] = pd.to_datetime(date['date'])

In [ ]:
date.info()

# 1.Extract year

In [ ]:
date['date_year'] =date['date'].dt.year
date.sample(5)

# 2. Extract Month

In [ ]:
date['date_month_no'] =date['date'].dt.month
date.head()

In [ ]:
date['date_month_name']=date['date'].dt.month_name()
date.head()

# Extract Days

In [ ]:
date['date_day']=date['date'].dt.day
date.head()

In [ ]:
# day of week
date['date_dow']=date['date'].dt.dayofweek
date.head()

In [ ]:
# day of week -name
date['date_dow_name']=date['date'].dt.day_name()
date.drop(columns=['product_id','city_id','orders']).head()

In [ ]:
date['date_is_weekend']=np.where(date['date_dow_name'].isin(['Saturday', 'Sunday']), 1, 0)
date.drop(columns=['product_id','city_id','orders']).head()

# Extract week of the year

In [ ]:
date['date_week'] =date['date'].dt.isocalendar().week
date.drop(columns=['product_id','city_id','orders']).head()

# Extract Quarter

In [ ]:
date['quarter'] =date['date'].dt.quarter
date.drop(columns=['product_id','city_id','orders']).head()

# Extract Semester

In [ ]:
date['semester'] =np.where(date['quarter'].isin([1,2]), 1, 2)
date.drop(columns=['product_id','city_id','orders']).head()



This line creates a new column `semester` based on the quarter number:

- `date['quarter'].isin([1,2])`
  - checks each row to see if quarter is 1 or 2
  - returns `True` for Q1 and Q2, `False` for Q3 and Q4

- `np.where(condition, 1, 2)`
  - if the condition is `True`, it returns `1`
  - if the condition is `False`, it returns `2`

So the full line means:

- if the quarter is 1 or 2, set `semester` to `1`
- otherwise set `semester` to `2`

In other words, it maps:
- Q1/Q2 → semester 1
- Q3/Q4 → semester 2

# Extract Time Elapsed between dates


In [ ]:
import datetime 
today = datetime.datetime.today()
today

Because in the working example `today` is a `datetime.datetime` object:

- `today = datetime.datetime.today()`

That is compatible with a pandas `datetime64[ns]` series, so pandas can subtract `date['date']` and return timedeltas.

In your failing example, `today` was a `datetime.date` object:

- `today = datetime.date.today()`

Pandas cannot subtract a plain Python `date` from a pandas datetime Series, so it raises the `TypeError`.

So the key difference is:

- `datetime.datetime` → works with pandas datetime Series
- `datetime.date` → does not work directly with pandas datetime Series

Created memory file [](file:///c%3A/Users/tanma/AppData/Roaming/Code/User/workspaceStorage/b880ecaae8d06d1e484a390e82558af0/GitHub.copilot-chat/memory-tool/memories/NzA1YTgyOWMtMDc5OS00MzhlLWJmZTUtZGZlZDZiNWViYTNi/date_vs_datetime_explanation.md)

## Why it works in one case and not the other

- `datetime.datetime.today()` returns a full timestamp with date and time.
- `datetime.date.today()` returns only a date without time.

Pandas datetime Series (`date['date']` after `pd.to_datetime`) can do arithmetic with `datetime.datetime` but not with plain `datetime.date`.

So:
- `datetime.datetime.today() - date['date']` works
- `datetime.date.today() - date['date']` raises `TypeError`

### Best fix
Use a pandas-compatible timestamp:

```python
today = pd.to_datetime(datetime.date.today())
```

or:

```python
today = pd.Timestamp.today().normalize()
```

Then `today - date['date']` will work correctly.

In [ ]:
today- date['date']

In [ ]:
date['date'].info()


In [ ]:
(today - date['date']).dt.days

In [ ]:
# Months passed

np.round((today -date['date']) / np.timedelta64(1, 'D')/30,0)

## Why this error happens

`np.timedelta64(1, 'M')` is not supported for timedelta arithmetic in NumPy/Pandas.

Allowed units are:
- `W`, `D`, `h`, `m`, `s`, `ms`, `us`, `ns`

So this line fails because `'M'` (month) is ambiguous in length.

## What to do

Use a supported timedelta unit, or calculate months differently.

### Option 1: approximate months as days
```python
np.round((today - date['date']) / np.timedelta64(1, 'D') / 30, 0)
```

### Option 2: compute month difference exactly
```python
today = pd.Timestamp.today().normalize()
months_passed = (today.year - date['date'].dt.year) * 12 + (today.month - date['date'].dt.month)
```

### Option 3: if you only need days
```python
(date['date'] - today).dt.days
```

Use one of these instead of `np.timedelta64(1, 'M')`.

# ------------------------------------------------------------------------------------------------------

## `np.timedelta64` explanation

- `np.timedelta64` creates a fixed-duration time interval in NumPy.
- Example: `np.timedelta64(1, 'D')` is one day, `np.timedelta64(1, 'h')` is one hour.
- In your code, `np.timedelta64(1, 'M')` is invalid because:
  - `'M'` means a month,
  - months have variable lengths (28–31 days),
  - NumPy only supports unambiguous units for timedeltas: `W`, `D`, `h`, `m`, `s`, `ms`, `us`, `ns`.

So this line fails:
```python
np.round((today - date['date']) / np.timedelta64(1, 'M'), 0)
```

Because pandas cannot interpret `1 month` as a fixed timedelta.

## What that code is trying to do

- `today - date['date']` produces a timedelta series
- dividing by `np.timedelta64(1, unit)` converts that timedelta into units
- `np.round(..., 0)` rounds the result to the nearest whole number

For months, use a different approach:
- approximate with days:
  ```python
  np.round((today - date['date']) / np.timedelta64(1, 'D') / 30, 0)
  ```
- or calculate exact months from year/month:
  ```python
  months_passed = (today.year - date['date'].dt.year) * 12 + (today.month - date['date'].dt.month)
  ```

## Notes on the later section

- `datetime.datetime.today()` returns a full timestamp with date and time.
- `datetime.date.today()` returns only a calendar date.
- A pandas datetime column created with `pd.to_datetime(...)` works with `datetime.datetime`, but not with plain `datetime.date`.

So the note means:
- `today = datetime.datetime.today()` is compatible with `date['date']`
- `today = datetime.date.today()` is not compatible for subtraction from a pandas datetime Series

That is why the working example uses `datetime.datetime.today()` and not `datetime.date.today()`.

In [50]:
time.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        1000 non-null   object
 1   product_id  1000 non-null   int64 
 2   city_id     1000 non-null   int64 
 3   orders      1000 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 31.4+ KB


In [51]:
# Converting to datetime datatype
time['date'] = pd.to_datetime(time['date'])

In [52]:
time.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1000 non-null   datetime64[ns]
 1   product_id  1000 non-null   int64         
 2   city_id     1000 non-null   int64         
 3   orders      1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3)
memory usage: 31.4 KB


In [53]:
time['hour'] = time['date'].dt.hour
time['min'] = time['date'].dt.minute
time['sec'] = time['date'].dt.second

time.head()

,date,product_id,city_id,orders,hour,min,sec
0,2019-12-10,5628,25,3,0,0,0
1,2018-08-15,3646,14,157,0,0,0
2,2018-10-23,1859,25,1,0,0,0
3,2019-08-17,7292,25,1,0,0,0
4,2019-01-06,4344,25,3,0,0,0


# Time difference

In [ ]:
today - time['date']

In [54]:
# in seconds

(today - time['date'])/np.timedelta64(1,'s')

0      207360000.0
1      249004800.0
2      243043200.0
3      217296000.0
4      236563200.0
          ...     
995    244339200.0
996    239241600.0
997    226108800.0
998    231724800.0
999    212198400.0
Name: date, Length: 1000, dtype: float64